<a href="https://colab.research.google.com/github/boyerdan2000-hipr/WarpX-PIC-Beam-Plasma-Wakefield-Propulsion/blob/main/hipr_picmi_dynamic_gradient_compressed_beam.py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==============================================================================
# CELL 1: WARPX ENVIRONMENT SETUP & FRESH CUDA COMPILATION
# ==============================================================================
import os
import sys

# 1. Install necessary system dependencies for compilation
!apt-get update -y
!apt-get install -y cmake g++ libopenmpi-dev openmpi-bin libfftw3-dev

# 2. Clone the WarpX repository if it is not present in the current runtime
if not os.path.exists('/content/WarpX'):
    !git clone https://github.com/ECP-WarpX/WarpX.git /content/WarpX

# 3. Wipe any old architecture builds to guarantee a fresh L4 compilation
!rm -rf /content/WarpX/build
!mkdir -p /content/WarpX/build

# 4. Configure CMake for CUDA (AMReX will automatically detect the L4 architecture)
%cd /content/WarpX/build
!cmake -DWarpX_COMPUTE=CUDA -DWarpX_PYTHON=ON ..

# 5. Compile WarpX (using 4 CPU threads to speed up the process)
!make -j 4

# 6. Append the newly compiled Python bindings to the system path
sys.path.insert(0, '/content/WarpX/build/lib')
sys.path.insert(0, '/content/WarpX/build/bin')

print("\n===================================================================")
print("   WarpX Compilation Complete! L4 GPU bindings are now active.")
print("===================================================================")

# Return to the root directory for subsequent cells
%cd /content

In [ ]:
# ==============================================================================
# CELL 2: WARPX PICMI INITIALIZATION (STATIONARY LITHIUM BASELINE - 30 PACKETS)
# ==============================================================================
import os
import sys
import numpy as np

sys.path.insert(0, '/content/WarpX/build/lib')
sys.path.insert(0, '/content/WarpX/build/bin')
from pywarpx import picmi

# 1. Constants & Densities
c = 299792458.0
m_p = 1.67262192e-27
e_charge = 1.602176634e-19

# Baseline Plasma Density
n_plasma = 1.0e19

# 2. Domain & Grid setup (Full 32mm Chamber, Stationary Grid)
z_min, z_max, r_max = -30.0e-3, 2.0e-3, 1.5e-3
grid = picmi.CylindricalGrid(
    number_of_cells=[256, 2048],
    lower_bound=[0, z_min],
    upper_bound=[r_max, z_max],
    lower_boundary_conditions=['none', 'dirichlet'],
    upper_boundary_conditions=['dirichlet', 'dirichlet'],
    lower_boundary_conditions_particles=['none', 'absorbing'],
    upper_boundary_conditions_particles=['absorbing', 'absorbing'],
    n_azimuthal_modes=1
)
solver = picmi.ElectromagneticSolver(grid=grid, method='Yee', cfl=0.99)
plasma_layout = picmi.GriddedLayout(n_macroparticle_per_cell=[2, 2, 1], grid=grid)

# 3. Plasma Distributions (Lithium Background - Stationary)
lithium_distribution = picmi.UniformDistribution(
    density=n_plasma,
    directed_velocity=[0.0, 0.0, 0.0]
)
electron_distribution = picmi.UniformDistribution(
    density=n_plasma,
    directed_velocity=[0.0, 0.0, 0.0]
)

electrons = picmi.Species(
    particle_type='electron',
    name='electrons',
    initial_distribution=electron_distribution
)

# Lithium-7 Ion Background (Li+)
lithium_ions = picmi.Species(
    particle_type='Li',
    charge_state=1,
    mass=7.016 * m_p,
    name='lithium_ions',
    initial_distribution=lithium_distribution
)

# 4. Define Radially Compressed Gold Beam (Bane-Chen Optimized Doorstep)
v_b = 0.2 * c
gamma = 1.02062
u_z = v_b * gamma

beam_radius = 500e-6
packet_length = 1.055e-3
period = 2.11e-3
I_peak = 6.0
beam_area = np.pi * (beam_radius**2)
n0 = I_peak / (e_charge * beam_area * v_b)

k_p = 2.0 * np.pi / period
local_z = f"(-z - {period}*floor(-z/{period}))"
step_fraction = 0.15
doorstep_shape = f"({step_fraction} + (1.0 - {step_fraction}) * (exp({k_p} * {local_z}) - 1) / (exp({k_p} * {packet_length}) - 1))"

doorstep_train = f"{n0} * (x < {beam_radius}) * (z > -21.1e-3) * (z <= 0.0) * " \
                 f"({local_z} < {packet_length}) * {doorstep_shape}"

gold_distribution = picmi.AnalyticDistribution(
    density_expression=doorstep_train,
    momentum_expressions=['0', '0', f"{u_z}"]
)

# Gold-197 Driver (Au51+)
gold_beam = picmi.Species(
    particle_type='Au',
    charge_state=51,
    mass=196.97 * m_p,
    name='gold_beam',
    initial_distribution=gold_distribution
)

# 5. Initialize Simulation
sim = picmi.Simulation(solver=solver, max_steps=120000, verbose=1)
sim.add_species(electrons, layout=plasma_layout)
sim.add_species(lithium_ions, layout=plasma_layout)
sim.add_species(gold_beam, layout=picmi.GriddedLayout(n_macroparticle_per_cell=[2, 4, 1], grid=grid))

print(f"Cell 2 Complete: 0.2c Au51+ driver initialized in 1.0e19 m^-3 Li+ plasma.")

In [ ]:
# ==============================================================================
# CELL 3: WARPX PICMI 120,000-STEP RUN (STATIONARY LITHIUM BASELINE)
# ==============================================================================
%matplotlib inline
import os
import sys
import numpy as np
import time
from datetime import datetime
import pytz

# 1. MOUNT GOOGLE DRIVE
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
save_dir = '/content/drive/My Drive/WarpX_Checkpoints/'

# Create directory if it doesn't exist (Bypassing rmtree to prevent API hang)
os.makedirs(save_dir, exist_ok=True)

stats_file = os.path.join(save_dir, 'raw_stats.txt')
with open(stats_file, 'w') as f:
    f.write("step,max_ez,min_ez,avg_abs_ez\n")

sys.path.insert(0, '/content/WarpX/build/lib')
sys.path.insert(0, '/content/WarpX/build/bin')
from pywarpx import picmi, fields, warpx

# 2. OVERRIDES & TIMING
tz_pdt = pytz.timezone('America/Los_Angeles')
start_time = datetime.now(tz_pdt)
print(f"Baseline Simulation started at: {start_time.strftime('%Y-%m-%d %H:%M:%S PDT')}")
print("Running 120,000 steps (30 Au51+ packets)...")

window_size = 10
kernel = np.ones(window_size) / window_size
loop_start_time = time.time()

# 3. 30-PACKET EXECUTION LOOP
for chunk in range(30):
    chunk_start_time = time.time()

    # Run 4,000 steps per chunk
    sim.step(4000)
    step_count = (chunk + 1) * 4000

    # Extract longitudinal electric field
    ez_grid = sim.fields.get('Efield_fp', dir='z', level=0)[...]

    if ez_grid.shape[0] > ez_grid.shape[1]:
        ez_on_axis = ez_grid[:, 0]
    else:
        ez_on_axis = ez_grid[0, :]

    Ez_MVm_current_chunk = ez_on_axis / 1e6

    # Calculate chunk statistics
    chunk_max = np.max(Ez_MVm_current_chunk)
    chunk_min = np.min(Ez_MVm_current_chunk)
    chunk_avg_abs = np.mean(np.abs(Ez_MVm_current_chunk))

    # Append statistics to text file
    with open(stats_file, 'a') as f:
        f.write(f"{step_count},{chunk_max},{chunk_min},{chunk_avg_abs}\n")

    # Apply moving average and downsample for storage efficiency
    local_moving_avg = np.convolve(Ez_MVm_current_chunk, kernel, mode='same')
    reduced_chunk = local_moving_avg[::10].astype(np.float32)

    # Save the chunk to Google Drive
    chunk_filename = f"{save_dir}ez_chunk_{step_count}.npy"
    np.save(chunk_filename, reduced_chunk)

    # Clear variables to prevent RAM bloat
    del ez_grid
    del ez_on_axis
    del Ez_MVm_current_chunk
    del local_moving_avg
    del reduced_chunk

    # Timing and Print Status
    chunk_elapsed = time.time() - chunk_start_time
    current_time_str = datetime.now(tz_pdt).strftime('%H:%M:%S PDT')

    print(f"[{current_time_str}] --> Au51+ Packet {chunk + 1}/30 injected. Step {step_count} written to Drive. (Chunk time: {chunk_elapsed/60:.1f} min)")

total_elapsed = time.time() - loop_start_time
print(f"Simulation Complete! Total Execution Time: {total_elapsed/60:.1f} minutes.")

In [ ]:
# ==============================================================================
# Cell 4: Data Analysis and Plotting (Au51+ / Li+ Baseline)
# ==============================================================================
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt
from google.colab import drive

# 1. Mount Drive
drive.mount('/content/drive')
save_dir = '/content/drive/My Drive/WarpX_Checkpoints/'

print("Scanning Google Drive for saved checkpoints...")

# 2. Load exact raw AC statistics
stats_file = os.path.join(save_dir, 'raw_stats.txt')
if os.path.exists(stats_file):
    stats_df = pd.read_csv(stats_file)
    global_max_ez = stats_df['max_ez'].max()
    global_min_ez = stats_df['min_ez'].min()
    global_avg_abs_ez = stats_df['avg_abs_ez'].mean()
else:
    print("Warning: raw_stats.txt not found. Metrics will be skipped.")
    global_max_ez, global_min_ez, global_avg_abs_ez = 0, 0, 0

# 3. Locate, sort, and stitch the compressed array chunks
chunk_files = glob.glob(os.path.join(save_dir, 'ez_chunk_*.npy'))
chunk_files.sort(key=lambda x: int(x.split('_chunk_')[-1].split('.npy')[0]))

print(f"Successfully loaded {len(chunk_files)} snapshot chunks.")
compressed_snapshots = [np.load(f) for f in chunk_files]
final_snapshot = compressed_snapshots[-1]
final_step = int(chunk_files[-1].split('_chunk_')[-1].split('.npy')[0])

# 4. Reconstruct the spatial axis based on the FULL STATIONARY GRID
z_min, z_max = -30.0e-3, 2.0e-3
original_num_cells = 2048
original_dz = (z_max - z_min) / original_num_cells

stride = 10
new_dz = original_dz * stride
z_array_compressed = np.linspace(z_min, z_max, len(final_snapshot)) * 1e3 # mm

# 5. Apply Butterworth low-pass filter
fs = 1.0 / new_dz
nyquist = 0.5 * fs
cutoff = 0.1 * nyquist
b, a = butter(4, cutoff / nyquist, btype='low', analog=False)
Ez_dc_filtered = filtfilt(b, a, final_snapshot)


# ==============================================================================
# PLOT 1: Raw Unfiltered Wakefield Gradient with Target Lines
# ==============================================================================
plt.figure(figsize=(12, 5))
plt.plot(z_array_compressed, final_snapshot, label='Simulated $E_z$ Gradient', color='blue', linewidth=1)

# Target Lines
plt.axhline(100, color='red', linestyle='--', label='100 MV/m Peak Target')
plt.axhline(-100, color='red', linestyle='--')
plt.axhline(50, color='green', linestyle=':', label='50 MV/m Avg Target')
plt.axhline(-50, color='green', linestyle=':')
plt.axhline(0, color='black', linewidth=1)

plt.xlim(np.min(z_array_compressed), np.max(z_array_compressed))
plt.xlabel('Longitudinal Position, z (mm)', fontsize=12)
plt.ylabel('Electric Field, $E_z$ (MV/m)', fontsize=12)
plt.title('Longitudinal Wakefield Gradient ($E_z$)\n0.2c Au51+ Continuous Train', fontsize=14, fontweight='bold')
plt.legend(loc='upper right')
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()


# ==============================================================================
# PLOT 2: 1D Macroscopic Ambipolar Towing Field
# ==============================================================================
plt.figure(figsize=(12, 6))
plt.plot(z_array_compressed, final_snapshot, label='Downsampled Wakefield ($E_z$)', color='blue', alpha=0.25)
plt.plot(z_array_compressed, Ez_dc_filtered, label='Macroscopic DC Ambipolar Field', color='red', linewidth=3)
plt.axhline(0, color='black', linewidth=1)
plt.xlim(np.min(z_array_compressed), np.max(z_array_compressed))
plt.xlabel('Longitudinal Position, z (mm)', fontsize=12)
plt.ylabel('Electric Field, $E_z$ (MV/m)', fontsize=12)
plt.title(f'Macroscopic Ponderomotive Acceleration & Ambipolar Towing\nLow-Pass Filtered Longitudinal Wakefield at Step {final_step:,}', fontsize=14, fontweight='bold')
plt.legend(loc='upper right')
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()


# ==============================================================================
# PLOT 3: 2D Spatiotemporal Evolution Heatmap
# ==============================================================================
evolution_matrix = np.vstack(compressed_snapshots)
step_array = np.arange(1, len(compressed_snapshots) + 1) * 4000

plt.figure(figsize=(14, 8))
limit = np.percentile(np.abs(evolution_matrix), 95)

pcm = plt.pcolormesh(
    z_array_compressed,
    step_array,
    evolution_matrix,
    cmap='RdBu_r',
    vmin=-limit,
    vmax=limit,
    shading='auto'
)

cbar = plt.colorbar(pcm, pad=0.02)
cbar.set_label('Electric Field, $E_z$ (MV/m)', fontsize=12)
plt.xlabel('Longitudinal Position, z (mm)', fontsize=12)
plt.ylabel('Simulation Step', fontsize=12)
plt.title(f'Spatiotemporal Evolution of the Longitudinal Wakefield\nContinuous Au51+ Train Injection (0 to {final_step:,} Steps)', fontsize=14, fontweight='bold')
plt.xlim(np.min(z_array_compressed), np.max(z_array_compressed))
plt.ylim(np.min(step_array), np.max(step_array))
plt.grid(True, linestyle='--', color='black', alpha=0.2)
plt.tight_layout()
plt.show()


# ==============================================================================
# PRINT FINAL METRICS
# ==============================================================================
print("\n==========================================================")
print("   0.2c Au51+ WAKEFIELD GRADIENT STATISTICS")
print("==========================================================")
print(f"Maximum Accelerating Gradient : {global_max_ez:8.2f} MV/m")
print(f"Maximum Decelerating Gradient : {global_min_ez:8.2f} MV/m")
print(f"Average Absolute Gradient     : {global_avg_abs_ez:8.2f} MV/m")
print("==========================================================\n")

print("==========================================================")
print("   MACROSCOPIC AMBIPOLAR TOWING FIELD (DC FILTERED)")
print("==========================================================")
print(f"Max DC Accelerating Gradient  : {np.max(Ez_dc_filtered):8.2f} MV/m")
print(f"Max DC Decelerating Gradient  : {np.min(Ez_dc_filtered):8.2f} MV/m")
print(f"Average Net Forward Pressure  : {np.mean(Ez_dc_filtered):8.2f} MV/m")
print("==========================================================")

In [ ]:
# ==============================================================================
# CELL 5: LONGITUDINAL PHASE SPACE EXTRACTION (LITHIUM BASELINE)
# ==============================================================================
import numpy as np
import matplotlib.pyplot as plt
from pywarpx.particle_containers import ParticleContainerWrapper

print("Extracting Lithium ion arrays from active AMReX memory...")

# Target the lithium ions
li_pc = ParticleContainerWrapper('lithium_ions')
z_li = np.concatenate(li_pc.get_particle_z(copy_to_host=True))
uz_li = np.concatenate(li_pc.get_particle_uz(copy_to_host=True))

stride = 10
z_plot = z_li[::stride]
vz_plot = uz_li[::stride]

plt.figure(figsize=(12, 6))
plt.scatter(z_plot * 1e3, vz_plot, s=4, alpha=0.7, color='green', edgecolors='none')

plt.axhline(0, color='black', linewidth=1, linestyle='-')
plt.xlabel('Longitudinal Position, z (mm)', fontsize=12)
plt.ylabel('Longitudinal Velocity, $v_z$ (m/s)', fontsize=12)
plt.title('Lithium Ion Phase Space: Macroscopic Forward Towing\nat Step 120,000', fontsize=14, fontweight='bold')
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()